In [ ]:
# ==============================================================================
# 1. IMPORTS (ALL NECESSARY LIBRARIES)
# ==============================================================================
import numpy as np
import pandas as pd

# Model Selection & Tuning
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold

# Preprocessing & Pipeline
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import (StandardScaler, MinMaxScaler, RobustScaler, 
                                   OneHotEncoder, OrdinalEncoder, PowerTransformer)

# Supervised Models (Classification & Regression)
from sklearn.linear_model import LinearRegression, LogisticRegression, SGDRegressor
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.svm import SVC, SVR

# Unsupervised Models
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA

# Evaluation Metrics
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, 
                             roc_auc_score, f1_score, mean_squared_error, r2_score)

import warnings
warnings.filterwarnings('ignore')

# ==============================================================================
# 2. DATA LOADING & TRAIN-TEST SPLIT (ALWAYS SPLIT FIRST!)
# ==============================================================================
# Load your dataset
df = pd.read_csv('your_dataset.csv')

# Separate Features (X) and Target (y)
X = df.drop('Target_Column', axis=1)
y = df['Target_Column']

# Train-Test Split (stratify=y for classification to keep class balance) [1, 2]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    random_state=42, 
    stratify=y # Remove stratify=y if it is a Regression problem
)

# ==============================================================================
# 3. PREPROCESSING PIPELINES & COLUMN TRANSFORMER [3, 4]
# ==============================================================================
# Define column types based on your dataset
num_cols = ['Age', 'Fare']                  # Continuous numbers
cat_nom_cols = ['Sex', 'Embarked']          # Categories without order
cat_ord_cols = ['Pclass', 'Education']      # Categories with order
passthru_cols = ['IsAlone']                 # Already 0/1, pass as is

# A. Numerical Pipeline: Impute Median -> Handle Skewness (Yeo-Johnson) -> Scale
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')), 
    ('transformer', PowerTransformer(method='yeo-johnson')), # Good for skewed data
    ('scaler', StandardScaler())
])

# B. Nominal Categorical Pipeline: Impute Mode -> One-Hot Encode (drop='first' to avoid dummy trap)
cat_nom_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore'))
])

# C. Ordinal Categorical Pipeline: Impute Mode -> Ordinal Encode
cat_ord_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(categories=[['Low', 'Medium', 'High']], # Replace with your categories
                               handle_unknown='use_encoded_value', unknown_value=-1))
])

# D. Combine everything in ColumnTransformer
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat_nom', cat_nom_pipeline, cat_nom_cols),
    ('cat_ord', cat_ord_pipeline, cat_ord_cols),
    ('passthru', 'passthrough', passthru_cols)
], remainder='drop') # 'drop' removes unspecified columns, 'passthrough' keeps them [5]

# ==============================================================================
# 4. MODEL DICTIONARY WITH KEY PARAMETERS [6, 7]
# ==============================================================================
# Choose the model you need for your exam from this dictionary
classification_models = {
    'Logistic_Regression': LogisticRegression(penalty='l2', C=1.0, class_weight='balanced', solver='lbfgs', max_iter=1000, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5, weights='distance', metric='minkowski', p=2),
    'Decision_Tree': DecisionTreeClassifier(criterion='gini', max_depth=5, min_samples_split=2, min_samples_leaf=1, random_state=42),
    'Random_Forest': RandomForestClassifier(n_estimators=100, max_depth=None, max_features='sqrt', class_weight='balanced', n_jobs=-1, random_state=42),
    'SVM': SVC(C=1.0, kernel='rbf', gamma='scale', probability=True, random_state=42)
}

regression_models = {
    'Linear_Regression': LinearRegression(),
    'SGD_Regressor': SGDRegressor(loss='squared_error', penalty='l2', alpha=0.0001, learning_rate='invscaling', max_iter=1000, random_state=42),
    'Random_Forest_Regressor': RandomForestRegressor(n_estimators=100, max_depth=None, max_features=1.0, n_jobs=-1, random_state=42),
    'SVR': SVR(C=1.0, kernel='rbf', gamma='scale', epsilon=0.1)
}

# ==============================================================================
# 5. FULL PIPELINE CREATION & TRAINING [8]
# ==============================================================================
# Example: Using Random Forest Classifier
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', classification_models['Random_Forest'])
])

# Fit on Train, Predict on Test
full_pipeline.fit(X_train, y_train)
y_pred = full_pipeline.predict(X_test)
y_prob = full_pipeline.predict_proba(X_test)[:, 1] # For ROC-AUC

# ==============================================================================
# 6. HYPERPARAMETER TUNING USING GridSearchCV [9, 10]
# ==============================================================================
# Use step_name__parameter_name syntax
param_grid = {
    'preprocessor__num__imputer__strategy': ['mean', 'median'],
    'model__n_estimators': [11-13],
    'model__max_depth': [5, 10, None],
    'model__min_samples_split': [14, 15]
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=full_pipeline,
    param_grid=param_grid,
    cv=skf, 
    scoring='accuracy', # Use 'f1' or 'roc_auc' for imbalanced data
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best CV Score: {grid_search.best_score_:.4f}")

# Predict using the best model found by GridSearch
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)

# ==============================================================================
# 7. EVALUATION METRICS [16, 17]
# ==============================================================================
# --- For Classification ---
print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred_best))

print("Accuracy:", accuracy_score(y_test, y_pred_best))
print("F1-Score:", f1_score(y_test, y_pred_best))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_best))

# --- For Regression (If you were doing regression) ---
# mse = mean_squared_error(y_test, y_pred_reg)
# rmse = np.sqrt(mse)
# r2 = r2_score(y_test, y_pred_reg)
# print(f"RMSE: {rmse:.4f}, R2: {r2:.4f}")


# ==============================================================================
# 8. UNSUPERVISED LEARNING CHEAT SHEET [18-20]
# ==============================================================================

# --- A. PCA (Principal Component Analysis) ---
# Note: Always scale data before PCA!
pca_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=0.95, random_state=42)) # Keeps 95% of variance
])
X_pca = pca_pipeline.fit_transform(X_train)
# print("Explained Variance Ratio:", pca_pipeline.named_steps['pca'].explained_variance_ratio_)


# --- B. K-Means Clustering ---
kmeans_pipeline = Pipeline([
    ('scaler', StandardScaler()), # K-Means is distance-based, MUST scale!
    ('kmeans', KMeans(n_clusters=3, init='k-means++', n_init=10, max_iter=300, random_state=42))
])
clusters = kmeans_pipeline.fit_predict(X_train)
# centroids = kmeans_pipeline.named_steps['kmeans'].cluster_centers_


# --- C. DBSCAN Clustering ---
# Note: DBSCAN does not have a predict() method for new data!
dbscan = DBSCAN(eps=0.5, min_samples=5, metric='euclidean')
# Must scale before applying DBSCAN
X_scaled = StandardScaler().fit_transform(X_train)
dbscan_labels = dbscan.fit_predict(X_scaled) 
# Labels: 0, 1, 2... are clusters. -1 means Noise/Outlier.